# DSL — building contracts from a string/dict with `parse_contract`

**Topic:** the contract DSL — declare a capability contract as a plain `dict` (or load it from JSON) instead of importing each value object.  
**Public API:** `paxman.parse_contract(spec) -> Contract`

This notebook follows the **reference template** (`00_email.ipynb`). Run every cell in order; no hidden state. It pairs with `10_engine.ipynb` — together they cover the two layers adopters most often want: *how do I declare a contract?* (this notebook) and *how do I pin authority editions?* (the Engine notebook).

> The DSL is a **closed vocabulary**: `kind` is a fixed set (`canonical_email`, `canonical_uuid`, …). An unknown `kind` or bad field raises `ContractError` **at parse time** — before any canonicalization runs. The contract is the truth (mandate Law 5): it declares *what* the canonical form is, never *how* it is produced.

In [ ]:
from paxman import parse_contract, canonicalize, ContractError

def show(spec, raw):
    """Parse a DSL spec, canonicalize `raw` against it, print status + value."""
    contract = parse_contract(spec)
    art = canonicalize(raw, contract)
    print(f"{spec!s:52} -> {art.status.name:14} {art.value!r}")
    return contract

## The `kind` vocabulary

Every built-in capability has a `canonical_<name>` kind. `parse_contract` is registry-driven, so this list is the single source of truth:

In [ ]:
from paxman._registry.contract_registry import known_kinds
print("supported kinds:")
for k in sorted(known_kinds()):
    print("  -", k)

## Parse -> canonicalize round-trip

Build the contract from a dict, then canonicalize. No need to import `Email`, `UUID`, etc.

In [ ]:
show({"kind": "canonical_email"}, "A@B.COM")
show({"kind": "canonical_uuid"}, "f47ac10b-58cc-4372-a567-0e02b2c3d479")
show({"kind": "canonical_date"}, "2024-01-02")
show({"kind": "canonical_country"}, "United States")
show({"kind": "canonical_url"}, "HTTP://EXAMPLE.COM/A")
show({"kind": "canonical_ip"}, "2001:0db8:0000:0000:0000:0000:0000:0001")
show({"kind": "canonical_boolean"}, "yes")
show({"kind": "canonical_phone"}, "+12025550123")
show({"kind": "canonical_geolocation"}, "40.7128N 74.0060W")

## Field variations on a kind

A kind may carry policy fields. These mirror the keyword args on the value-object factory (e.g. `Email(provider_aliases=...)`, `Money(currency=...)`).

In [ ]:
# email with Gmail aliasing (dot-strip + tag removal)
show({"kind": "canonical_email", "provider_aliases": "gmail"}, "John.Doe+spam@gmail.COM")

# money REQUIRES a currency field
show({"kind": "canonical_money", "currency": "USD"}, "USD 12.50")

## Errors — `ContractError` at parse time

Bad `kind`, missing `kind`, non-dict input, or an invalid field value all raise `ContractError` **before** canonicalize runs. The orchestrator turns an unknown *kind* into `Status.UNSUPPORTED`; everything else is a construction error you catch directly.

In [ ]:
for bad in [
    {"kind": "email"},            # wrong key form (must be canonical_*)
    {"kind": "widget"},           # not a real kind
    {"nokind": 1},                 # missing kind
    "not-a-dict",                 # must be a dict
    {"kind": "canonical_money", "currency": "XYZ"},  # unknown currency
]:
    try:
        parse_contract(bad)
        print("unexpectedly OK:", bad)
    except ContractError as exc:
        print(f"ContractError({bad!r}): {str(exc)[:55]}")

## A parsed contract is the source of truth

`parse_contract` short-circuits on an already-parsed contract: passing a contract value object back in returns it unchanged (mandate Law 5). So the DSL and the value-object factories are two doors to the same room.

In [ ]:
from paxman import Email
c = Email()                       # value-object form
c2 = parse_contract(c)            # DSL form with a contract input
print("same object returned:", c is c2)
print("canonicalize still works:", canonicalize("A@B.COM", c2).value)

## Where to go next

- **`10_engine.ipynb`** — pin authority editions with `Engine.with_authorities` / `authority_override`.
- Capability notebooks `00_email.ipynb` … `09_uuid.ipynb` — the data domains, with more input variants each.
- Background: `NOTEBOOK_INPUTS.md` (verified per-capability inputs) and `ARCHITECTURE.md` (the contract-is-truth model).